### From ELIZA to Statistical Models

**ELIZA's limitation:** Hand-crafted rules, no learning from data

**Next step:** What if we could learn patterns automatically from text?

Enter **statistical language modeling** - the foundation of everything from Mark V. Shaney to modern LLMs.

In [ ]:
import nltk
from nltk.chat import eliza

# Download required NLTK data (only needed first time)
try:
    nltk.download('punkt', quiet=True)
    print("✓ NLTK data downloaded")
except:
    print("! NLTK data already available")

# Initialize ELIZA chatbot
eliza_chatbot = eliza.eliza_chatbot
print("🤖 ELIZA chatbot initialized")
print("💡 ELIZA uses pattern matching to simulate a therapist")
print()

# Show some of ELIZA's patterns
print("Example ELIZA patterns:")
patterns = [
    "I am sad → Why are you sad?",
    "My mother → Tell me more about your mother", 
    "I feel → What makes you feel that way?",
    "Everyone → Surely not everyone",
    "Always → Can you think of a specific example?"
]

for pattern in patterns:
    print(f"  • {pattern}")

print()
print("=" * 50)
print("INTERACTIVE DEMO: Chat with ELIZA")
print("=" * 50)
print("Try saying things like:")
print("  'I am worried about my grades'")
print("  'My family doesn't understand me'")  
print("  'I feel like nobody listens'")
print("Type 'quit' to exit the conversation")
print()

# Interactive demo section
conversation_count = 0
max_exchanges = 8  # Limit for notebook environment

while conversation_count < max_exchanges:
    try:
        user_input = input("You: ").strip()
        
        if user_input.lower() in ['quit', 'exit', 'bye', 'goodbye']:
            print("ELIZA: Goodbye. It was nice talking to you.")
            break
        
        if not user_input:
            print("ELIZA: Please tell me something.")
            continue
            
        # Get ELIZA's response
        eliza_response = eliza_chatbot.respond(user_input)
        print(f"ELIZA: {eliza_response}")
        print()
        
        conversation_count += 1
        
    except (EOFError, KeyboardInterrupt):
        print("\nELIZA: It seems our session has ended. Take care.")
        break

if conversation_count >= max_exchanges:
    print(f"ELIZA: We've had a good {max_exchanges}-exchange session. Thank you for talking.")

print()
print("🔍 ANALYSIS: What just happened?")
print("• ELIZA used simple pattern matching - no real understanding")
print("• Yet it probably felt somewhat natural and responsive")
print("• This demonstrates the 'ELIZA effect' - humans readily attribute")
print("  understanding to systems that just manipulate symbols")
print("• Modern LLMs are vastly more sophisticated but work on similar principles:")
print("  pattern recognition in text, just at massive scale")

# What is LLM? From Shannon to Persuasion

**PSAM 3707 - April 22, 2026**

This notebook takes you from the simplest language models to modern LLM persuasion. Three parts:

1. **Small Language Model**: Shannon's Markov chain trained on Bible/Shakespeare/Wikipedia
2. **Mark V. Shaney**: Classic OSS implementation on same texts
3. **Paper Replication**: Reproduce findings from Costello, Williams & Ceci, or Lin et al.

---

## Setup: Text Sources

Choose your training text: Bible, Shakespeare, or any Wikipedia topic.

In [ ]:
import requests
import re
import random
from collections import defaultdict, Counter
from bs4 import BeautifulSoup
import numpy as np
import matplotlib.pyplot as plt

def get_bible_text():
    """Download King James Bible from Project Gutenberg"""
    url = "https://www.gutenberg.org/files/10/10-0.txt"
    response = requests.get(url)
    text = response.text
    # Extract just the biblical text (remove Gutenberg headers/footers)
    start = text.find("The First Book of Moses")
    end = text.find("End of the Project Gutenberg EBook")
    return text[start:end] if start != -1 and end != -1 else text

def get_shakespeare_text():
    """Download Shakespeare complete works from Project Gutenberg"""
    url = "https://www.gutenberg.org/files/100/100-0.txt"
    response = requests.get(url)
    text = response.text
    # Extract main content
    start = text.find("THE SONNETS")
    end = text.find("End of the Project Gutenberg EBook")
    return text[start:end] if start != -1 and end != -1 else text

def get_wikipedia_text(topic):
    """Download Wikipedia page for given topic"""
    url = f"https://en.wikipedia.org/wiki/{topic.replace(' ', '_')}"
    response = requests.get(url)
    soup = BeautifulSoup(response.content, 'html.parser')
    
    # Extract main content paragraphs
    content = soup.find('div', {'id': 'mw-content-text'})
    if content:
        paragraphs = content.find_all('p')
        text = ' '.join([p.get_text() for p in paragraphs])
        return text
    return "Could not fetch Wikipedia content for: " + topic

# Choose your text source
TEXT_CHOICE = input("Choose text source (bible/shakespeare/wikipedia): ").lower().strip()

if TEXT_CHOICE == "bible":
    raw_text = get_bible_text()
    print(f"Loaded Bible: {len(raw_text):,} characters")
elif TEXT_CHOICE == "shakespeare":
    raw_text = get_shakespeare_text()
    print(f"Loaded Shakespeare: {len(raw_text):,} characters")
elif TEXT_CHOICE == "wikipedia":
    topic = input("Enter Wikipedia topic: ")
    raw_text = get_wikipedia_text(topic)
    print(f"Loaded Wikipedia '{topic}': {len(raw_text):,} characters")
else:
    # Default to Shakespeare
    raw_text = get_shakespeare_text()
    print(f"Defaulting to Shakespeare: {len(raw_text):,} characters")

# Preview the text
print("\nFirst 200 characters:")
print(repr(raw_text[:200]))

## Part 1: Small Language Model (Shannon's Markov Chain)

Build a 1-state Markov model: predict next word based on current word.

In [ ]:
def preprocess_text(text):
    """Clean and tokenize text into words"""
    # Convert to lowercase and split into words
    text = re.sub(r'[^a-zA-Z\s]', '', text.lower())
    words = text.split()
    return [word for word in words if word]  # Remove empty strings

def build_markov_chain(words, order=1):
    """Build n-gram Markov chain from word list"""
    chain = defaultdict(list)
    
    for i in range(len(words) - order):
        # Create n-gram key
        if order == 1:
            key = words[i]
        else:
            key = tuple(words[i:i+order])
        
        # Next word is the value
        next_word = words[i + order]
        chain[key].append(next_word)
    
    return dict(chain)

def calculate_shannon_entropy(chain):
    """Calculate Shannon entropy of the language model"""
    total_entropy = 0
    total_contexts = 0
    
    for context, next_words in chain.items():
        if len(next_words) == 0:
            continue
            
        # Count frequency of each next word
        word_counts = Counter(next_words)
        total_words = len(next_words)
        
        # Calculate entropy for this context
        context_entropy = 0
        for count in word_counts.values():
            prob = count / total_words
            if prob > 0:
                context_entropy -= prob * np.log2(prob)
        
        total_entropy += context_entropy
        total_contexts += 1
    
    return total_entropy / total_contexts if total_contexts > 0 else 0

# Build 1-state Markov chain
words = preprocess_text(raw_text)
print(f"Processed into {len(words):,} words")
print(f"Vocabulary size: {len(set(words)):,} unique words")

markov_1 = build_markov_chain(words, order=1)
entropy_1 = calculate_shannon_entropy(markov_1)

print(f"\n1-state Markov chain:")
print(f"  Contexts: {len(markov_1):,}")
print(f"  Shannon entropy: {entropy_1:.3f} bits per word")
print(f"  Perplexity: {2**entropy_1:.1f}")

# Show example transitions
print("\nExample word transitions:")
sample_words = random.sample(list(markov_1.keys()), min(5, len(markov_1)))
for word in sample_words:
    next_words = markov_1[word][:10]  # First 10 possible next words
    print(f"  '{word}' → {next_words}")

In [ ]:
def generate_text_markov(chain, start_word=None, length=50):
    """Generate text using Markov chain"""
    if not chain:
        return "No chain available"
    
    # Pick random starting word if none provided
    if start_word is None or start_word not in chain:
        start_word = random.choice(list(chain.keys()))
    
    result = [start_word]
    current_word = start_word
    
    for _ in range(length - 1):
        if current_word in chain and chain[current_word]:
            next_word = random.choice(chain[current_word])
            result.append(next_word)
            current_word = next_word
        else:
            # Dead end - pick new random word
            current_word = random.choice(list(chain.keys()))
            result.append(current_word)
    
    return ' '.join(result)

# Generate sample text
print("Sample text from 1-state Markov chain:")
for i in range(3):
    sample = generate_text_markov(markov_1, length=30)
    print(f"  {i+1}. {sample}")

## ELIZA (1966): The Illusion of Understanding

Before we move to Mark V. Shaney, let's meet ELIZA - the first chatbot that convinced people it understood them.

**Joseph Weizenbaum's ELIZA:** A pattern-matching chatbot that played therapist
- Used simple rules: "I am X" → "Why are you X?"
- No real understanding, just clever text manipulation
- **The shocking result:** People thought it genuinely understood them

**Connection to course themes:** Early attempt at persuasive conversation through the illusion of empathy.

**Source code:** https://github.com/nltk/nltk/blob/develop/nltk/chat/eliza.py

## Part 2: Mark V. Shaney Implementation

Classic OSS Markov text generator. Uses 2-state (trigram) model for better coherence.

In [ ]:
class MarkVShaney:
    """Mark V. Shaney text generator (classic OSS implementation style)"""
    
    def __init__(self, order=2):
        self.order = order
        self.chain = defaultdict(list)
        self.starters = []  # Good starting n-grams
    
    def train(self, text):
        """Train on text corpus"""
        words = preprocess_text(text)
        
        # Build n-gram chain
        for i in range(len(words) - self.order):
            key = tuple(words[i:i+self.order])
            next_word = words[i + self.order]
            self.chain[key].append(next_word)
            
            # Collect sentence starters (capitalize first word)
            if i == 0 or words[i-1].endswith(('.', '!', '?')):
                self.starters.append(key)
        
        print(f"Trained Mark V. Shaney on {len(words):,} words")
        print(f"Generated {len(self.chain):,} {self.order}-gram contexts")
        print(f"Found {len(self.starters):,} potential sentence starters")
    
    def generate(self, length=50, start_key=None):
        """Generate text using the chain"""
        if not self.chain:
            return "Model not trained"
        
        # Pick starting n-gram
        if start_key is None:
            if self.starters:
                current_key = random.choice(self.starters)
            else:
                current_key = random.choice(list(self.chain.keys()))
        else:
            current_key = start_key
        
        result = list(current_key)
        
        for _ in range(length - self.order):
            if current_key in self.chain and self.chain[current_key]:
                next_word = random.choice(self.chain[current_key])
                result.append(next_word)
                
                # Slide the window
                current_key = current_key[1:] + (next_word,)
            else:
                # Dead end - restart
                if self.starters:
                    current_key = random.choice(self.starters)
                else:
                    current_key = random.choice(list(self.chain.keys()))
                result.extend(current_key)
        
        return ' '.join(result)
    
    def get_entropy(self):
        """Calculate model entropy"""
        return calculate_shannon_entropy(self.chain)

# Train Mark V. Shaney
shaney = MarkVShaney(order=2)
shaney.train(raw_text)

entropy_2 = shaney.get_entropy()
print(f"\n2-state Markov entropy: {entropy_2:.3f} bits per word")
print(f"Perplexity: {2**entropy_2:.1f}")
print(f"Improvement over 1-state: {entropy_1 - entropy_2:.3f} bits")

# Generate samples
print("\nMark V. Shaney generated text:")
for i in range(3):
    sample = shaney.generate(length=40)
    print(f"  {i+1}. {sample}")

## Comparison: 1-state vs 2-state Models

See how adding context improves text quality and reduces uncertainty.

In [ ]:
# Build different order models for comparison
orders = [1, 2, 3]
entropies = []
perplexities = []

for order in orders:
    if order == 1:
        chain = markov_1
        entropy = entropy_1
    else:
        chain = build_markov_chain(words, order=order)
        entropy = calculate_shannon_entropy(chain)
    
    entropies.append(entropy)
    perplexities.append(2**entropy)
    print(f"{order}-gram model: {entropy:.3f} bits, perplexity {2**entropy:.1f}")

# Plot entropy vs model order
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(orders, entropies, 'bo-', linewidth=2, markersize=8)
plt.xlabel('Model Order (n-gram)')
plt.ylabel('Shannon Entropy (bits per word)')
plt.title('Model Complexity vs Uncertainty')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(orders, perplexities, 'ro-', linewidth=2, markersize=8)
plt.xlabel('Model Order (n-gram)')
plt.ylabel('Perplexity')
plt.title('Model Complexity vs Perplexity')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nKey insight: More context → Lower entropy → Better prediction")
print(f"This is why LLMs with huge context windows work so well for persuasion!")

## Part 3: Paper Replication

Replicate findings from the assigned papers using simple language models or LLM APIs.

In [ ]:
# Choose which paper to replicate
PAPERS = {
    "costello": "Costello et al. (2024) - Conspiracy belief reduction",
    "williams": "Williams & Ceci (2026) - Biased writing assistants", 
    "lin": "Lin et al. (2025) - Political persuasion dialogues"
}

print("Choose paper to replicate:")
for key, title in PAPERS.items():
    print(f"  {key}: {title}")

paper_choice = input("\nEnter choice (costello/williams/lin): ").lower().strip()

if paper_choice not in PAPERS:
    paper_choice = "costello"  # Default
    print(f"Defaulting to: {PAPERS[paper_choice]}")
else:
    print(f"Selected: {PAPERS[paper_choice]}")

In [ ]:
if paper_choice == "costello":
    print("🔬 Replicating Costello et al. (2024): Conspiracy Belief Reduction\n")
    
    # Simulate conspiracy beliefs and evidence-based counterarguments
    conspiracy_beliefs = [
        "The moon landing was faked by Hollywood",
        "Vaccines contain mind control chips", 
        "Climate change is a hoax by scientists",
        "9/11 was an inside job by the government",
        "COVID-19 was created in a lab intentionally"
    ]
    
    # Generate evidence-based responses using our Markov model
    # (In real study, this was GPT-4 Turbo with actual evidence)
    def generate_counterargument(belief, model):
        """Generate a counterargument using our language model"""
        # Simple approach: generate text and score for evidence-like content
        response = model.generate(length=30)
        
        # Mock "persuasiveness score" based on text properties
        evidence_words = ['evidence', 'study', 'research', 'data', 'fact', 'proof', 'scientific']
        persuasiveness = sum(1 for word in evidence_words if word in response.lower())
        
        return response, persuasiveness
    
    print("Simulated conspiracy belief reduction experiment:")
    print("(Note: Real study used GPT-4 Turbo with actual evidence)\n")
    
    total_initial_belief = 0
    total_final_belief = 0
    
    for i, belief in enumerate(conspiracy_beliefs, 1):
        print(f"{i}. Conspiracy belief: \"{belief}\"")
        
        # Simulate initial belief strength (1-10 scale)
        initial_belief = random.randint(6, 10)  # Strong initial belief
        
        # Generate counterargument
        counter_text, persuasiveness = generate_counterargument(belief, shaney)
        
        # Simulate belief reduction based on "persuasiveness"
        reduction = min(persuasiveness, 4)  # Max reduction of 4 points
        final_belief = max(1, initial_belief - reduction)
        
        print(f"   Initial belief strength: {initial_belief}/10")
        print(f"   Generated counter: {counter_text[:60]}...")
        print(f"   Persuasiveness score: {persuasiveness}")
        print(f"   Final belief strength: {final_belief}/10")
        print(f"   Reduction: {initial_belief - final_belief} points\n")
        
        total_initial_belief += initial_belief
        total_final_belief += final_belief
    
    # Calculate overall effect
    avg_initial = total_initial_belief / len(conspiracy_beliefs)
    avg_final = total_final_belief / len(conspiracy_beliefs)
    effect_size = (avg_initial - avg_final) / avg_initial * 100
    
    print(f"📊 RESULTS SUMMARY:")
    print(f"   Average initial belief: {avg_initial:.1f}/10")
    print(f"   Average final belief: {avg_final:.1f}/10")
    print(f"   Belief reduction: {effect_size:.1f}%")
    print(f"   ") 
    print(f"   🎯 Costello et al. found ~20% reduction")
    print(f"   Our simulation: {effect_size:.1f}% reduction")
    
elif paper_choice == "williams":
    print("🔬 Replicating Williams & Ceci (2026): Biased Writing Assistants\n")
    
    # Simulate biased autocomplete suggestions
    writing_prompts = [
        "Immigration policy should",
        "Climate change is", 
        "Gun control laws",
        "Healthcare systems",
        "Economic inequality"
    ]
    
    def generate_biased_completions(prompt, bias_direction):
        """Generate biased completions for writing prompts"""
        # Use our Markov model to generate completions
        completions = []
        for _ in range(3):
            completion = shaney.generate(length=15)
            completions.append(completion)
        
        # Mock bias scoring
        if bias_direction == "conservative":
            bias_words = ['tradition', 'family', 'security', 'freedom', 'order']
        else:  # liberal
            bias_words = ['equality', 'progress', 'justice', 'inclusive', 'reform']
        
        bias_scores = []
        for completion in completions:
            score = sum(1 for word in bias_words if word in completion.lower())
            bias_scores.append(score)
        
        return completions, bias_scores
    
    print("Simulated biased writing assistant experiment:")
    print("(Note: Real study used actual AI writing tools)\n")
    
    for i, prompt in enumerate(writing_prompts, 1):
        print(f"{i}. Writing prompt: \"{prompt}\"")
        
        # Simulate both bias conditions
        for bias in ['conservative', 'liberal']:
            completions, scores = generate_biased_completions(prompt, bias)
            avg_bias = np.mean(scores)
            
            print(f"   {bias.title()} bias condition:")
            print(f"     Completions: {completions[0][:40]}...")
            print(f"     Bias score: {avg_bias:.1f}")
        print()
    
    print(f"📊 RESULTS SUMMARY:")
    print(f"   🎯 Williams & Ceci found 20% attitude shift")
    print(f"   Key finding: Users unaware of bias influence")
    print(f"   Mechanism: Real-time suggestion during writing")

elif paper_choice == "lin":
    print("🔬 Replicating Lin et al. (2025): Political Persuasion Dialogues\n")
    
    # Simulate political candidate preferences
    candidates = ["Candidate A", "Candidate B"]
    countries = ["US", "Canada", "Poland"]
    
    def simulate_political_dialogue(candidate, country, model):
        """Simulate AI political persuasion dialogue"""
        # Generate political argument
        argument = model.generate(length=25)
        
        # Mock persuasiveness based on text properties
        persuasive_words = ['economy', 'jobs', 'security', 'future', 'change', 'better']
        persuasiveness = sum(1 for word in persuasive_words if word in argument.lower())
        
        # Simulate preference shift (Lin et al. found 3.9-10 percentage points)
        base_preference = 50  # 50% baseline support
        shift = min(persuasiveness * 1.5, 8)  # Max 8 point shift
        new_preference = base_preference + shift
        
        return argument, shift, new_preference
    
    print("Simulated cross-national political persuasion experiment:")
    print("(Note: Real study used actual political chatbots)\n")
    
    total_shifts = []
    
    for country in countries:
        print(f"🌍 {country} Election Simulation:")
        for candidate in candidates:
            argument, shift, new_pref = simulate_political_dialogue(candidate, country, shaney)
            total_shifts.append(shift)
            
            print(f"   {candidate}: {argument[:50]}...")
            print(f"     Preference shift: +{shift:.1f} percentage points")
            print(f"     New support level: {new_pref:.1f}%")
        print()
    
    avg_shift = np.mean(total_shifts)
    print(f"📊 RESULTS SUMMARY:")
    print(f"   Average preference shift: {avg_shift:.1f} percentage points")
    print(f"   🎯 Lin et al. found 3.9-10 point shifts")
    print(f"   Key finding: Cross-national effectiveness")
    print(f"   Mechanism: Factual claims > emotional manipulation")

print(f"\n✅ Paper replication complete!")
print(f"Note: This is a simplified simulation. Real studies used:")
print(f"- Actual LLM APIs (GPT-4, etc.)")
print(f"- Real human participants")
print(f"- Rigorous experimental controls")
print(f"- Pre-registered hypotheses")

## Summary: From Simple Models to LLM Persuasion

**What we learned:**

1. **Small Language Models** (Shannon/Markov): Basic text prediction using statistical patterns
2. **Mark V. Shaney**: Classic improvement showing how more context reduces uncertainty  
3. **Modern LLMs**: Scale these principles to billions of parameters and massive context windows

**Key insights for persuasion:**
- **More context → Better prediction → More personalized arguments**
- **Scale enables capabilities**: GPT-4 can do what Markov chains cannot
- **Foundation models**: Pre-trained on all text → adaptable to any persuasion task

**Connection to April 20 (Embeddings):**
- Recommender systems: Embeddings capture user similarity
- Language models: Embeddings capture semantic meaning
- **Both enable personalization at scale**

**For the final exam, remember:**
- Shannon entropy measures prediction difficulty
- Context reduces uncertainty (key to LLM success)
- Scaling laws: bigger models → better persuasion
- Technical capabilities enable the effects in Costello, Williams, and Lin papers